# Visual Classifier — Fine-Tuning
Fine-tunes `dima806/ai_vs_human_generated_image_detection` on
`julienlucas/midjourney-dalle-sd-nanobananapro-dataset` and saves the result to `./fine_tuned_model`.

> This notebook is **self-contained** — it can be run locally or on Google Colab without needing any local `.py` files.

Run **`visual_classifier_evaluation.ipynb`** afterwards to compare the original vs. fine-tuned model.

## 0. Install Dependencies (Colab only)
Skip this cell if running locally with your own conda/venv environment.

In [1]:
import sys

# Only install on Colab (or any non-local environment)
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q transformers datasets scikit-learn accelerate

## 1. Imports

In [2]:
import warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoImageProcessor,
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"MPS available   : {torch.backends.mps.is_available()}")

PyTorch version : 2.10.0+cu128
CUDA available  : True
MPS available   : False


## 2. Configuration
Adjust these as needed before running.

In [3]:
MODEL_NAME    = "dima806/ai_vs_human_generated_image_detection"
DATASET_NAME  = "julienlucas/midjourney-dalle-sd-nanobananapro-dataset"
OUTPUT_DIR    = "./fine_tuned_model"

EPOCHS        = 3
BATCH_SIZE    = 16
LEARNING_RATE = 2e-5
GRAD_ACCUM    = 4       # effective batch size = BATCH_SIZE * GRAD_ACCUM

## 3. Helper: Metrics Function

In [4]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average='binary'
    )
    acc = accuracy_score(labels, predictions)
    return {'accuracy': acc, 'f1': f1, 'precision': precision, 'recall': recall}

## 4. Load Dataset & Model

In [5]:
print(f"Loading dataset: {DATASET_NAME}")
dataset = load_dataset(DATASET_NAME)
train_ds = dataset['train']
test_ds  = dataset['test']
print(f"Train: {len(train_ds)} samples  |  Test: {len(test_ds)} samples")

Loading dataset: julienlucas/midjourney-dalle-sd-nanobananapro-dataset


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00009.parquet:   0%|          | 0.00/497M [00:00<?, ?B/s]

data/train-00001-of-00009.parquet:   0%|          | 0.00/499M [00:00<?, ?B/s]

data/train-00002-of-00009.parquet:   0%|          | 0.00/283M [00:00<?, ?B/s]

...

data/test-00000-of-00002.parquet:   0%|          | 0.00/346M [00:00<?, ?B/s]

data/test-00001-of-00002.parquet:   0%|          | 0.00/203M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10695 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Train: 10695 samples  |  Test: 2000 samples


In [6]:
print(f"Loading base model: {MODEL_NAME}")
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model     = AutoModelForImageClassification.from_pretrained(
    MODEL_NAME, ignore_mismatched_sizes=True
)
print("Model loaded.")

Loading base model: dima806/ai_vs_human_generated_image_detection


preprocessor_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/737 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Model loaded.


## 5. Apply Transforms

In [7]:
def transforms(examples):
    inputs = processor(
        [img.convert("RGB") for img in examples["image"]],
        return_tensors="pt"
    )
    inputs["labels"] = examples["label"]
    return inputs

print("Applying transformations...")
train_ds.set_transform(transforms)
test_ds.set_transform(transforms)
print("Done.")

Applying transformations...
Done.


## 6. Training Arguments

In [9]:
# Determine whether to enable MPS (Apple Silicon) — ignored on Colab/CUDA
use_mps = torch.backends.mps.is_available()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    remove_unused_columns=False,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    warmup_steps=100,
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
)

def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    labels       = torch.tensor([example["labels"]       for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}

## 7. Train
> ⏱️ Estimated time: ~35 min on M4 MacBook (MPS) | ~10–15 min on Colab T4 GPU

In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=collate_fn,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=processor,
    compute_metrics=compute_metrics,
)

print("Starting training...")
train_results = trainer.train()

print("\nSaving model...")
trainer.save_model(OUTPUT_DIR)
trainer.log_metrics("train", train_results.metrics)
trainer.save_metrics("train", train_results.metrics)
trainer.save_state()

print(f"\n✅ Fine-tuned model saved to: {OUTPUT_DIR}")

Starting training...


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.397695,0.394203,0.818000,0.810417,0.845652,0.778000
2,0.213101,0.312517,0.871000,0.873900,0.854685,0.894000
3,0.138346,0.291172,0.879500,0.880040,0.876115,0.884000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saving model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

***** train metrics *****
  epoch                    =          3.0
  total_flos               = 2315575710GF
  train_loss               =       0.4248
  train_runtime            =   0:36:19.26
  train_samples_per_second =       14.723
  train_steps_per_second   =        0.231

✅ Fine-tuned model saved to: ./fine_tuned_model


## 8. (Colab only) Download the Fine-Tuned Model
If running on Colab, zip and download the model so you can use it locally.

In [12]:
import shutil, os

if IN_COLAB:
    from google.colab import drive

    # Mount Google Drive (prompts for auth once)
    drive.mount('/content/drive')

    DRIVE_SAVE_DIR = '/content/drive/MyDrive/fine_tuned_model'

    # Zip the model folder
    print('Zipping model...')
    zip_path = shutil.make_archive('fine_tuned_model', 'zip', OUTPUT_DIR)
    size_gb  = os.path.getsize(zip_path) / 1e9
    print(f'Archive: {zip_path}  ({size_gb:.2f} GB)')

    # Copy zip to Google Drive
    os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
    dest = os.path.join(DRIVE_SAVE_DIR, 'fine_tuned_model.zip')
    shutil.copy(zip_path, dest)

    print(f'\n✅ Saved to Google Drive: {dest}')
    print('Go to drive.google.com → MyDrive/fine_tuned_model/ to download it.')
else:
    print('Not on Colab — model already saved locally at:', OUTPUT_DIR)

Mounted at /content/drive
Zipping model...
Archive: /content/fine_tuned_model.zip  (3.17 GB)

✅ Saved to Google Drive: /content/drive/MyDrive/fine_tuned_model/fine_tuned_model.zip
Go to drive.google.com → MyDrive/fine_tuned_model/ to download it.
